In [ ]:
# ...existing code...
import pandas as pd
import numpy as np

df = pd.read_csv("creditcard.csv")

# Velocity feature: count of same Amount within the last 1 hour (3600s), including current txn
df["_row_id"] = np.arange(len(df), dtype=np.int64)

tmp = df[["_row_id", "Time", "Amount"]].sort_values("Time").reset_index(drop=True)
tmp["Transaction_Count_1H"] = 0

for amt, g in tmp.groupby("Amount", sort=False):
    t = g["Time"].to_numpy()
    left = np.searchsorted(t, t - 3600, side="left")
    right = np.arange(len(t)) + 1
    counts = right - left
    tmp.loc[g.index, "Transaction_Count_1H"] = counts

df = df.merge(tmp[["_row_id", "Transaction_Count_1H"]], on="_row_id", how="left")
df.drop(columns=["_row_id"], inplace=True)
df["Transaction_Count_1H"] = df["Transaction_Count_1H"].astype(np.int16)

print("Added feature: Transaction_Count_1H")
print(df[["Time", "Amount", "Transaction_Count_1H"]].head())
# ...existing code...

In [ ]:
#this counts a data cleaning since we are removing redundant data 
# df2=df[df["Class"]==1]
# 
# print(df2.duplicated().sum())
# 

df = df.drop_duplicates()

print(len(df[df["Class"] == 1]))
print(len(df[df["Class"] == 0]))

print((473/284807) * (100))

print(df.duplicated().sum())

In [ ]:
df.dtypes #lazy one

In [ ]:
print("nof of frauds are ")
print(df[df["Class"]==1].shape)
print("nof of no frauds are ")
print(df[df["Class"]==0].shape)



In [ ]:
import math
x = df["Time"].to_list()
print(max(x))

x = [math.floor((i / 3600) % 24) for i in x]

print(max(x))

y = df["Class"].to_list()


In [ ]:
time_fraud = {}
for i in range(24):
    time_fraud[i] = 0

# zip is a built in function that allows us to iterate over two lists at the same time
# here we are iterating over the time and fraud lists at the same time and counting the number of frauds for each hour of the day
for time, fraud in zip(x, y):
    if fraud == 1:
        time_fraud[time] = time_fraud[time] + 1   # Changed x → time

for time, fraud in time_fraud.items():
    print(f'{time} : {fraud}')

In [ ]:
import matplotlib.pyplot as plt

hours = list(time_fraud.keys())
fraud_counts = list(time_fraud.values())

plt.plot(hours, fraud_counts, marker='o')  
plt.xlabel('Hour of Day (0-23)')
plt.ylabel('Number of Fraudulent Transactions')
plt.title('Fraud Counts by Hour')
plt.xticks(range(0, 24, 2)) 
plt.grid(True)
plt.show()

# Drop the Time col 

In [ ]:
cols=df.columns
print(cols)
for i in cols:
   print((df[i].isnull().sum()))

In [ ]:
print(df)

#okay so i 249 fraud data points

In [ ]:
import seaborn as sns
cor=df.corr()
sns.heatmap(cor)

#can u guys see any useful info from this ?

In [ ]:
cols=df.columns

print(cols)
tot=0
ctot=0
for i in cols[1:-2]:

     tot+=df[i].var()
for i in cols[1:-2]:
     print(i+" "+str(df[i].var()/tot))

for i in cols[1:-2]:
     ctot+=df[i].var()/tot
     print(i+" "+str(ctot))
# Explains how PCA ouptput should look like -> Higher total ratio to the lowest also before
# it's wrong didn't include the v1.. 

In [ ]:
cols[0]

In [ ]:
# comment this out 

# import matplotlib.pyplot as plt

# clas = [float(0), float(1)]
# colors = ['red', 'blue']      

# for j in range(1, len(cols) - 2):
#     for i, k in zip(clas, colors):
#         subset = df[df['Class'] == i]
#         print(subset.shape)
#         plt.scatter(subset[cols[j]], subset[cols[j+1]], color=k, label=f"Class {i}")      

#   plt.xlabel(cols[j])
#   plt.ylabel(cols[j+1])
#   plt.legend()
#   plt.show()



#what do u guys  infer fomr the plots ?
#one things for sure we cant use clustering of any kind

we can't visually separate fraud and not fraud one beacuse it's enclosed no clear separation.

In [ ]:
import matplotlib.pyplot as plt

df.groupby('Class')['Time'].plot(kind='hist')
plt.legend(['Normal','Fraud'])
plt.show()

# Does this plot gave you any idea?? -> [Dharhas]

In [ ]:
import seaborn as sns

sns.boxplot(x='Class', y='Amount', data=df)

In [ ]:
sns.boxplot(x='Class', y='Amount', data=df)
plt.yscale('log')
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

pcs = [f'V{i}' for i in range(1, 29)]

for pc in pcs:
    plt.figure(figsize=(5,4))
    sns.boxplot(x='Class', y=pc, data=df)
    plt.title(f"{pc} distribution by Class")
    plt.show()

#**Again here i dont find any good inference **
**Mostly all class 1 are clustered so we use that one as plus over class 0 **
ut i wnt hink much info gain from that



**What does this  mean do fraud transaction happen ony using less amount ??**

isn't it obvious from the above box plot that fraud happens only at the low amounts..

In [ ]:
# It performs downsampling of the majority class (non-fraudulent transactions) 
# to balance the dataset with the minority class (fraudulent transactions).
from sklearn.utils import resample

df_majority = df[df.Class == 0]
df_minority = df[df.Class == 1]

df_majority_downsampled = resample(df_majority,
                                  replace=False,
                                  n_samples=1000,
                                  random_state=42)

df_balanced = pd.concat([df_majority_downsampled, df_minority])
df_balanced["Class"].value_counts()


Model Deployment and Evaluation

In [ ]:
# It defines a function `undersample_train` that takes in features `X` and labels `y`,
#  separates the majority and minority classes, and performs undersampling on the
#  majority class to create a balanced dataset. The function returns the balanced 
# features and labels.
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import resample
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc
import matplotlib.pyplot as plt

# Undersampling :- No. of Frauds = No. of Genuine Transactions

def undersample_train(X, y):
    X_majority = X[y == 0]
    X_minority = X[y == 1]
    y_majority = y[y == 0]
    y_minority = y[y == 1]
    n_minority = len(X_minority)
    if n_minority == 0:
        return X, y
    X_majority_down, y_majority_down = resample(
        X_majority, y_majority,
        replace=False,
        n_samples=n_minority,
        random_state=42
    )
    X_bal = np.vstack([X_majority_down, X_minority])
    y_bal = np.hstack([y_majority_down, y_minority])
    return X_bal, y_bal

# Precision of top k Outputs 
# y_scores is the model’s confidence ranking output, not the final class label.

def precision_at_k(y_true, y_scores, k):
    if len(y_scores) < k:
        k = len(y_scores)
    idx = np.argsort(y_scores)[::-1][:k]
    return np.mean(y_true[idx])

In [ ]:
# Split Data Chronologically -> 80% Test and 20% Train

df = df.sort_values('Time')
feature_cols = [col for col in df.columns if col not in ['Time', 'Class']]
X = df[feature_cols].values
y = df['Class'].values

split_idx = int(0.8 * len(df))
X_train_raw, X_test = X[:split_idx], X[split_idx:]
y_train_raw, y_test = y[:split_idx], y[split_idx:]

print(f"Train size: {len(X_train_raw)}, Test size: {len(X_test)}")
print(f"Fraud % in train: {100*np.mean(y_train_raw):.4f}%, test: {100*np.mean(y_test):.4f}%")

In [ ]:
# Undersampled Data 

X_train_bal, y_train_bal = undersample_train(X_train_raw, y_train_raw)

print("After undersampling:")
print(f"Training set size: {len(X_train_bal)}")
print(f"Fraud % in balanced training set: {100 * np.mean(y_train_bal):.2f}%")
print(f"Number of fraud samples: {np.sum(y_train_bal)}")
print(f"Number of genuine samples: {np.sum(y_train_bal == 0)}")

In [ ]:
rf_baseline = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf_baseline.fit(X_train_bal, y_train_bal)

print("Baseline Random Forest training completed.")

In [ ]:
y_scores_rf = rf_baseline.predict_proba(X_test)[:, 1]

auc_rf = roc_auc_score(y_test, y_scores_rf)

k_values = [50, 100, 200, 500]
pk_results = {}
for k in k_values:
    pk_results[k] = precision_at_k(y_test, y_scores_rf, k)

print("=== Baseline Random Forest Results ===")
print(f"AUC: {auc_rf:.4f}")
for k in k_values:
    print(f"Precision@{k}: {pk_results[k]:.4f}")

In [ ]:
precision, recall, _ = precision_recall_curve(y_test, y_scores_rf)
pr_auc = auc(recall, precision)

plt.figure(figsize=(7,5))
plt.plot(recall, precision, label=f'RF Baseline (PR AUC = {pr_auc:.3f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision‑Recall Curve – Baseline Random Forest')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.bar([str(k) for k in k_values], list(pk_results.values()), color='skyblue')
plt.xlabel('k (number of alerts)')
plt.ylabel('Precision@k')
plt.title('Baseline Random Forest – Precision@k')
plt.ylim(0, 1)
for i, k in enumerate(k_values):
    plt.text(i, pk_results[k] + 0.02, f'{pk_results[k]:.3f}', ha='center')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

Approach 2 -> Captures and Updates on go with the new detected Frauds

In [ ]:
# Cell P1: Convert Time (seconds) into batch indices (4-hour batches)
seconds_per_hour = 3600
hours = df['Time'] / seconds_per_hour
batch_hours = 4          # each batch = 4 hours
df['batch'] = (hours // batch_hours).astype(int)

# Show the batch distribution
batches = sorted(df['batch'].unique())
print(f"Total batches (each {batch_hours} hours): {batches}")
print(f"Transactions per batch:\n{df['batch'].value_counts().sort_index()}")

In [ ]:
# Cell P2: Parameters for the realistic simulation (adapted for short time span)
delta_batches = 2         # verification latency = 2 batches = 8 hours (since 4h/batch)
k_alerts = 100            # number of alerts (transactions) per batch that investigators check
M_batches = 4             # number of batches of delayed samples to keep in training
Q_batches = M_batches + delta_batches   # number of batches of feedbacks to keep (4+2=6)

In [ ]:
# Cell P3: Use the first few batches as initial "delayed" data to train a provisional model
initial_batches = batches[:4]          # first 4 batches (first 16 hours)
initial_df = df[df['batch'].isin(initial_batches)]
X_init = initial_df[feature_cols].values
y_init = initial_df['Class'].values

# Balance the initial data by undersampling (same helper as baseline)
X_init_bal, y_init_bal = undersample_train(X_init, y_init)

# Train a provisional Random Forest
provisional_rf = RandomForestClassifier(n_estimators=100, random_state=42)
provisional_rf.fit(X_init_bal, y_init_bal)

print("Provisional model trained on initial batches (delayed-only).")

In [ ]:
# Cell P4: Loop through the next few batches, generate top‑k alerts as feedbacks
feedback_batches = batches[4:8]        # batches 5 to 8 (next 16 hours)
feedbacks_list = []
delayed_list = []

for b in feedback_batches:
    batch_df = df[df['batch'] == b]
    if len(batch_df) == 0:
        continue
    X_batch = batch_df[feature_cols].values
    y_batch = batch_df['Class'].values
    scores = provisional_rf.predict_proba(X_batch)[:, 1]   # fraud probability
    
    n_alerts = min(k_alerts, len(batch_df))
    idx_top = np.argsort(scores)[::-1][:n_alerts]         # indices of top‑k alerts
    feedbacks_list.append(batch_df.iloc[idx_top])         # these become feedbacks
    
    # The rest (non‑alerted) become delayed samples (will be used after latency)
    non_alert_idx = [i for i in range(len(batch_df)) if i not in idx_top]
    delayed_list.append(batch_df.iloc[non_alert_idx])

# Combine all feedbacks from these batches into one DataFrame
F_df = pd.concat(feedbacks_list, ignore_index=True)
# Combine all delayed samples from these batches
D_df = pd.concat(delayed_list, ignore_index=True)

print(f"Feedbacks collected: {len(F_df)} transactions")
print(f"Delayed samples collected: {len(D_df)} transactions")
print(f"Fraud rate in feedbacks: {100*F_df['Class'].mean():.2f}%")
print(f"Fraud rate in delayed: {100*D_df['Class'].mean():.2f}%")

In [ ]:
# Cell P5: Train two separate Random Forest models
# Model F: trained only on feedbacks (biased, recent)
X_F = F_df[feature_cols].values
y_F = F_df['Class'].values
X_F_bal, y_F_bal = undersample_train(X_F, y_F)
clf_F = RandomForestClassifier(n_estimators=100, random_state=42)
clf_F.fit(X_F_bal, y_F_bal)

# Model D: trained only on delayed samples (unbiased, older)
X_D = D_df[feature_cols].values
y_D = D_df['Class'].values
X_D_bal, y_D_bal = undersample_train(X_D, y_D)
clf_D = RandomForestClassifier(n_estimators=100, random_state=42)
clf_D.fit(X_D_bal, y_D_bal)

print("Training of F (feedback classifier) and D (delayed classifier) completed.")

In [ ]:
# Cell P6: Use the remaining batches (after feedback generation) as a test set
test_batches = batches[8:]            # batches 9 to end (last 16 hours)
if len(test_batches) == 0:
    # fallback: use last 20% of rows
    test_df = df.iloc[int(0.8*len(df)):]
else:
    test_df = df[df['batch'].isin(test_batches)]

X_test = test_df[feature_cols].values
y_test = test_df['Class'].values

# Get probabilities from both classifiers
p_F = clf_F.predict_proba(X_test)[:, 1]
p_D = clf_D.predict_proba(X_test)[:, 1]

# Aggregate with equal weight (α = 0.5 as in the paper)
alpha = 0.5
p_agg = alpha * p_F + (1 - alpha) * p_D

# Evaluate the aggregated model
auc_agg = roc_auc_score(y_test, p_agg)
print(f"Proposed Aggregated AUC: {auc_agg:.4f}")

k_values = [50, 100, 200]
for k in k_values:
    pk = precision_at_k(y_test, p_agg, k)
    print(f"Precision@{k}: {pk:.4f}")